# Jev classifier for Enron accounting emails

This notebook uses TypeSafe Jev 1.13 through OpenRouter's [Decisions API](https://openrouter.ai/docs/api/api-reference/alphadecisions/submit-a-decisions-questions-and-answers-request). Jev answers one typed yes/no question about each email and returns a probability. This is **zero-shot classification**, not model training. The curated test labels are used only to measure performance; the `search_phrase` and prior model predictions are never sent to Jev.

The notebook defaults to all 559 held-out emails. Set `MAX_TEST_ROWS = 100` for a smaller pilot. Requests may incur OpenRouter charges. Results are saved after every response so reruns resume. The model's 32K-token context and [current model details](https://openrouter.ai/typesafe/jev-1.13) should be checked before increasing the email length limit.


## 1. Set up access

Install Jupyter in your Python environment if needed. Before starting Jupyter, set `OPENROUTER_API_KEY` in that environment (for example, `$env:OPENROUTER_API_KEY = "..."` in PowerShell). Keep the key out of the notebook and source control.


In [ ]:
import os
from pathlib import Path
import sys
from collections import Counter

# Works when Jupyter starts from the repository root or its EDA folder.
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data' / 'enron_labeled_curated_test.csv').exists()), None)
if REPO is None:
    raise FileNotFoundError('Start Jupyter from the GTPracticum repository or a folder inside it')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.jev_openrouter import (
    MODEL, MAX_EMAIL_CHARS, QUESTION, evaluate, load_labeled_rows,
    run_classifier, sample_rows,
)

TEST_PATH = REPO / 'data' / 'enron_labeled_curated_test.csv'
OUTPUT_PATH = REPO / 'data' / 'jev_test_predictions.csv'
MAX_TEST_ROWS = None  # Set to 100 for a smaller random pilot.
SEED = 42

all_test_rows = load_labeled_rows(TEST_PATH)
selected_rows = sample_rows(all_test_rows, MAX_TEST_ROWS, seed=SEED)
print('Model:', MODEL)
print('Full test set:', len(all_test_rows), Counter(r['label'] for r in all_test_rows))
print('Selected:', len(selected_rows), Counter(r['label'] for r in selected_rows))
print('Results file:', OUTPUT_PATH)


## 2. Review the decision rule

Positive means substantive discussion or evidence of Enron accounting irregularities; relevant reporting and allegations count. Routine finance emails and unrelated messages are negative. Edit the question in `scripts/jev_openrouter.py` before running if your intended label meaning differs. A changed question will produce a configuration mismatch when resuming an existing results CSV, protecting the evaluation from mixed settings.


In [ ]:
QUESTION


## 3. Classify the selected held-out emails

This cell sends the `email` field to OpenRouter. It sends no labels. Jev returns a probability from 0 to 1; the predicted class uses a fixed 0.5 threshold. Emails longer than `MAX_EMAIL_CHARS` are clipped and flagged in the output. New API calls require `OPENROUTER_API_KEY`; reruns skip rows already saved with matching inputs and settings, so an existing complete run can be evaluated without a key.


In [ ]:
api_key = os.environ.get('OPENROUTER_API_KEY', '')  # Needed only for rows not already saved.

results = run_classifier(
    selected_rows,
    OUTPUT_PATH,
    api_key=api_key,
    model=MODEL,
    max_email_chars=MAX_EMAIL_CHARS,
)
print(f'Classified {len(results)} emails; saved to {OUTPUT_PATH}')


## 4. Evaluate

Metrics describe the selected rows. The default selection is the complete held-out test set. The saved CSV is reused; completed rows are skipped.


In [ ]:
metrics = evaluate(results)
for name, value in metrics.items():
    print(f'{name}: {value:.4f}' if isinstance(value, float) else f'{name}: {value}')
